# 工具变量教程

### 用 Card (1995) 的数据估计教育回报率

多读一年书，工资能提高多少？这是劳动经济学里最经典的问题之一，也是讲工具变量（IV）最常用的例子。Card (1995) 用「成长地附近有没有四年制大学」作为受教育年限的工具变量，数据随 StatsPAI 自带。

本教程不只是跑一条 `ivreg` 命令。我们会把 IV 估计量手工拆成第一阶段和简约式，看清楚它在算什么，再讨论弱工具变量、过度识别和排除限制这三个真正决定结果可信度的问题。全程离线。

| 节 | 内容 | 你会学到 |
| --- | --- | --- |
| 1 | 为什么 OLS 不够 | 内生性从哪里来 |
| 2 | 工具变量的三个条件 | 哪些能检验，哪些不能 |
| 3 | 数据 | 变量含义 |
| 4 | OLS 基准 | 参照点 |
| 5 | 手工拆解 IV | 第一阶段、简约式、Wald 比值，手工两步法的标准误为什么是错的 |
| 6 | `sp.ivreg` | 一行代码和它附带的诊断 |
| 7 | 弱工具变量 | 一个模拟实验，以及四个应对工具 |
| 8 | 过度识别 | 第二个工具变量、Hansen J 检验、LIML |
| 9 | 估计的是谁的效应 | LATE，以及 IV 为什么比 OLS 大 |
| 10 | 排除限制不完全成立怎么办 | 一个敏感性分析 |
| 11 | 小结 | 报告 IV 结果的清单 |

## 1. 为什么 OLS 不够

设想工资方程是

$$
\log(\text{wage}_i) = \alpha + \beta\,\text{educ}_i + X_i'\gamma + u_i .
$$

OLS 要得到 $\beta$ 的一致估计，需要 $\text{educ}$ 与误差项 $u$ 不相关。这个条件在这里很难成立，至少有两个原因。

- **遗漏变量**。能力、家庭背景、对未来的耐心，这些因素既影响一个人读多少书，也直接影响工资。它们观测不到，都留在 $u$ 里。如果能力强的人读书更多、工资也更高，OLS 会**高估** $\beta$。
- **测量误差**。问卷里自报的受教育年限有误差。经典测量误差会使 OLS 系数向 0 衰减，也就是**低估** $\beta$。

两种偏差方向相反，净效应不清楚。工具变量的思路是另找一个变量 $Z$，它能推动受教育年限变化，但与 $u$ 无关。只使用由 $Z$ 引起的那部分教育差异来估计 $\beta$。

## 2. 工具变量的三个条件

| 条件 | 含义 | 能否用数据检验 |
| --- | --- | --- |
| **相关性** | $Z$ 确实影响 $\text{educ}$ | 能。看第一阶段 |
| **外生性** | $Z$ 与工资方程里的不可观测因素无关 | 不能直接检验 |
| **排除限制** | $Z$ 只通过 $\text{educ}$ 影响工资，没有别的渠道 | 不能直接检验 |

后两条经常合称「工具变量有效」。它们靠的是研究设计和制度背景的论证，数据帮不上多少忙。

Card 的工具变量是 `nearc4`，表示 1966 年时居住的地方附近有没有四年制大学。相关性的论证很直接，家门口有大学，上学成本更低。外生性和排除限制则需要更多辩护。大学往往建在城市和较富裕的地区，这些地方工资本来就高。所以回归里要控制地区和城市变量，条件于这些变量之后，再假设 `nearc4` 与个人的不可观测因素无关。这个假设是否可信，读者可以自己判断，第 10 节会回到这个问题。

In [1]:
import warnings

warnings.filterwarnings("ignore")  # 教程里隐藏无关提示，正式分析时建议保留

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statspai as sp

plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 120)
print("StatsPAI 版本:", sp.__version__)

StatsPAI 版本: 1.39.2


## 3. 数据

`sp.datasets.card_1995()` 是 NLSYM 调查的真实数据，3,010 名男性。

| 变量 | 含义 | 角色 |
| --- | --- | --- |
| `lwage` | 1976 年小时工资的对数 | 结果变量 |
| `educ` | 受教育年限 | 内生解释变量 |
| `nearc4` | 成长地附近有四年制大学 | 工具变量 |
| `nearc2` | 成长地附近有两年制大学 | 第 8 节的第二个工具变量 |
| `exper`, `expersq` | 潜在工作经验及其平方 | 控制变量 |
| `black` | 黑人 | 控制变量 |
| `south` | 居住在南方 | 控制变量 |
| `smsa` | 居住在都市区 | 控制变量 |

In [2]:
card = sp.datasets.card_1995()
print(card.shape)
card.describe().T[["mean", "std", "min", "max"]].round(3)

(3010, 9)


,mean,std,min,max
lwage,6.262,0.444,4.605,7.785
educ,13.263,2.677,1.000,18.000
exper,8.856,4.142,0.000,23.000
expersq,95.579,84.618,0.000,529.000
black,0.234,0.423,0.000,1.000
south,0.404,0.491,0.000,1.000
smsa,0.713,0.452,0.000,1.000
nearc4,0.682,0.466,0.000,1.000
nearc2,0.441,0.497,0.000,1.000


In [3]:
# 工具变量把样本分成两组。先看两组的原始差别
card.groupby("nearc4")[["educ", "lwage", "black", "south", "smsa"]].mean().round(3)

,educ,lwage,black,south,smsa
nearc4,,,,,
0,12.698,6.155,0.280,0.563,0.479
1,13.527,6.311,0.212,0.329,0.822


附近有大学的人平均多读约 0.8 年书，工资也更高。同时他们更少住在南方、更多住在都市区。这提醒我们，`nearc4` 并不是随机分配的，地区变量必须控制。

## 4. OLS 基准

In [4]:
controls = "exper + expersq + black + south + smsa"

ols = sp.regress(f"lwage ~ educ + {controls}", data=card, robust="hc1")
print(f"OLS: educ 系数 = {ols.params['educ']:.4f}  (se {ols.std_errors['educ']:.4f})")

OLS: educ 系数 = 0.0740  (se 0.0036)


OLS 估计的教育回报率是每年 7.4%，标准误很小。接下来看 IV 给出什么。

## 5. 手工拆解 IV

只有一个内生变量、一个工具变量时，IV 估计量可以写成两个回归系数之比。

- **第一阶段**。内生变量对工具变量回归，$\text{educ} = \pi_0 + \pi_1 Z + X'\pi_2 + v$。
- **简约式**（reduced form）。结果变量直接对工具变量回归，$\log(\text{wage}) = \rho_0 + \rho_1 Z + X'\rho_2 + e$。
- **IV 估计量**。$\hat\beta_{IV} = \hat\rho_1 / \hat\pi_1$。

这个比值有很直观的解释。附近有大学使工资提高了 $\rho_1$，使教育年限增加了 $\pi_1$。如果工具变量只通过教育影响工资，那么每多一年教育带来的工资增长就是两者之比。

In [5]:
first_stage = sp.regress(f"educ ~ nearc4 + {controls}", data=card, robust="hc1")
reduced_form = sp.regress(f"lwage ~ nearc4 + {controls}", data=card, robust="hc1")

pi1 = first_stage.params["nearc4"]
rho1 = reduced_form.params["nearc4"]

print(f"第一阶段 : nearc4 使 educ 增加 {pi1:.4f} 年  (se {first_stage.std_errors['nearc4']:.4f})")
print(f"简约式   : nearc4 使 lwage 增加 {rho1:.4f}    (se {reduced_form.std_errors['nearc4']:.4f})")
print(f"Wald 比值: {rho1:.4f} / {pi1:.4f} = {rho1 / pi1:.4f}")

第一阶段 : nearc4 使 educ 增加 0.3373 年  (se 0.0806)
简约式   : nearc4 使 lwage 增加 0.0446    (se 0.0164)
Wald 比值: 0.0446 / 0.3373 = 0.1323


控制了地区和经验之后，附近有大学使教育年限增加约三分之一年，使工资提高约 4.5%。两者相除，IV 估计的教育回报率约为 13%，差不多是 OLS 的两倍。

「两阶段最小二乘」这个名字来自另一种等价的算法。第一步用第一阶段回归得到 $\widehat{\text{educ}}$，第二步用它替换原来的 `educ` 再跑 OLS。

In [6]:
# 手工两步法。系数是对的，标准误不能用
card["educ_hat"] = first_stage.predict(card)
manual = sp.regress(f"lwage ~ educ_hat + {controls}", data=card)

print(f"手工两步法: educ_hat 系数 = {manual.params['educ_hat']:.4f}  (se {manual.std_errors['educ_hat']:.5f})")

手工两步法: educ_hat 系数 = 0.1323  (se 0.05043)


系数和 Wald 比值完全一样。但这个标准误不能用。第二步回归的残差是 $y - \hat\beta\,\widehat{\text{educ}}$，而正确的残差应该是 $y - \hat\beta\,\text{educ}$，要用**原始**的内生变量来算。手工两步法用错了残差，残差方差和标准误也就跟着错了。

错多少取决于内生性的强弱。下面用一份内生性很强的模拟数据（真实系数为 1）把差别放大给你看。

In [7]:
rng = np.random.default_rng(1)
n = 2000
z = rng.normal(size=n)
u = rng.normal(size=n)
v = 0.8 * u + 0.6 * rng.normal(size=n)   # 两个方程的误差相关系数为 0.8
toy = pd.DataFrame({"z": z, "x": 0.5 * z + v})
toy["y"] = 1.0 * toy["x"] + u

toy["x_hat"] = sp.regress("x ~ z", data=toy).predict(toy)
two_step = sp.regress("y ~ x_hat", data=toy)
proper = sp.ivreg("y ~ (x ~ z)", data=toy)

print(f"手工两步法 : 系数 {two_step.params['x_hat']:.4f}   se {two_step.std_errors['x_hat']:.4f}")
print(f"sp.ivreg   : 系数 {proper.params['x']:.4f}   se {proper.std_errors['x']:.4f}")

手工两步法 : 系数 0.9386   se 0.0897
sp.ivreg   : 系数 0.9386   se 0.0494


系数相同，手工两步法的标准误却大了将近一倍。在 Card 的数据里两者碰巧很接近（下一节会看到正确的标准误），但你事先无法知道差别有多大，所以实际工作中不要手工做两步。

## 6. `sp.ivreg`

公式里用括号标出内生变量和工具变量，写法是 `(内生变量 ~ 工具变量)`，括号外是外生控制变量。相当于 Stata 的 `ivregress 2sls lwage exper expersq black south smsa (educ = nearc4), r small`。

In [8]:
iv = sp.ivreg(f"lwage ~ (educ ~ nearc4) + {controls}", data=card, robust="hc1")
print(iv.summary())

Model: IV-2SLS
Method: Two-Stage Least Squares
Dependent Variable: lwage
           Coefficient  Std. Error  t-statistic  P>|t|  [0.025  0.975]
Intercept       3.7528      0.8177       4.5894 0.0000  2.1495  5.3561
exper           0.1075      0.0211       5.0857 0.0000  0.0661  0.1489
expersq        -0.0023      0.0003      -6.5872 0.0000 -0.0030 -0.0016
black          -0.1308      0.0515      -2.5393 0.0112 -0.2318 -0.0298
south          -0.1049      0.0229      -4.5755 0.0000 -0.1499 -0.0599
smsa            0.1313      0.0298       4.4064 0.0000  0.0729  0.1898
educ            0.1323      0.0486       2.7232 0.0065  0.0370  0.2275

Model Diagnostics:
--------------------
R-squared                        : 0.2252
N instruments                    : 1
N endogenous                     : 1
First-stage F (educ)             : 17.5133
First-stage F p-value (educ)     : 0.0000
First-stage F, non-robust (educ) : 16.7176
Partial R² (educ)                : 0.0055
Hausman F-stat                  

系数 0.1323 与手工算的 Wald 比值一致，这里的标准误是按正确的残差计算的。

注意它和 OLS 的对比。IV 的标准误是 OLS 的十几倍。这是工具变量的固有代价，它只利用了 `educ` 变异中由 `nearc4` 引起的那一小部分。结果下方的 `Partial R²` 只有 0.0055，说明工具变量解释了教育年限残余变异的 0.55%。

结果表下面自动附带了一组诊断，后面几节逐个解释。

## 7. 弱工具变量

工具变量和内生变量的相关性如果很弱，2SLS 会出三个问题。

1. 估计量偏向 OLS，样本再大也消不掉。工具变量个数越多，这个偏差越严重。
2. 抽样分布不再接近正态，尾部很厚。
3. 常规的 t 检验和置信区间不可靠，覆盖率低于名义水平。

### 一个模拟实验

下面的数据生成过程里，真实系数是 1，内生性很强（两个方程的误差相关系数为 0.8）。只改变第一阶段系数的大小，看 2SLS 的表现。为了跑得快，这里直接用矩阵公式。

In [9]:
def weak_iv_experiment(pi, reps=4000, n=500, beta=1.0, rho=0.8, seed=0):
    """恰好识别的 IV 模型。pi 是第一阶段系数，越小工具变量越弱。"""
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(reps, n))
    u = rng.normal(size=(reps, n))
    v = rho * u + np.sqrt(1 - rho**2) * rng.normal(size=(reps, n))
    x = pi * z + v
    y = beta * x + u

    zx, zy, zz = (z * x).sum(1), (z * y).sum(1), (z * z).sum(1)
    b_iv = zy / zx
    b_ols = (x * y).sum(1) / (x * x).sum(1)

    # 2SLS 的常规 t 区间是否覆盖真值
    resid = y - b_iv[:, None] * x
    se_iv = np.sqrt((resid**2).mean(1) * zz) / np.abs(zx)
    cover_t = np.abs(b_iv - beta) < 1.96 * se_iv

    # Anderson-Rubin 检验。在真值处，y - beta*x 应该与 z 无关
    e0 = y - beta * x
    ar_t = (z * e0).sum(1) / np.sqrt((e0**2).mean(1) * zz)
    cover_ar = np.abs(ar_t) < 1.96

    # 第一阶段 F 统计量
    pi_hat = zx / zz
    v_hat = x - pi_hat[:, None] * z
    f_stat = pi_hat**2 * zz / (v_hat**2).mean(1)

    return {
        "first-stage coef": pi,
        "mean first-stage F": f_stat.mean(),
        "median 2SLS": np.median(b_iv),
        "mean OLS": b_ols.mean(),
        "t-interval coverage": cover_t.mean(),
        "AR coverage": cover_ar.mean(),
    }


mc = pd.DataFrame([weak_iv_experiment(pi) for pi in (0.5, 0.2, 0.1, 0.05)])
print("真实系数 = 1，名义覆盖率 = 0.95")
mc.round(3)

真实系数 = 1，名义覆盖率 = 0.95


,first-stage coef,mean first-stage F,median 2SLS,mean OLS,t-interval coverage,AR coverage
0,0.50,126.233,0.999,1.640,0.952,0.95
1,0.20,20.913,0.998,1.770,0.938,0.95
2,0.10,5.925,1.010,1.792,0.913,0.95
3,0.05,2.203,1.224,1.798,0.889,0.95


从上往下读这张表。

- 第一阶段 F 在 100 以上时，2SLS 的中位数就是真值，t 区间的覆盖率是 95%。
- F 约为 20 时，t 区间的覆盖率已经略低于 95%。F 降到 6 左右，覆盖率只有 91%。
- F 约为 2 时，2SLS 的中位数向 OLS 的方向移动了 0.2，覆盖率不到 89%。只有一个工具变量时 2SLS 近似中位数无偏，所以偏差要到 F 很小才显现。工具变量多于一个时情况会糟得多。
- 最后一列的 **Anderson-Rubin** 置信集在任何强度下都保持 95% 的覆盖率。

这就是为什么现在的文献建议，工具变量强度有疑问时，报告对弱工具变量稳健的置信区间。

### 回到 Card 的数据

`sp.estat(iv, "firststage")` 相当于 Stata 的 `estat firststage`。

In [10]:
_ = sp.estat(iv, "firststage")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  First-stage F-statistic (weak instruments)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  H0: Instruments are weak (excluded instruments have no explanatory power)

  F(1, 3003) =      17.5133
  p-value    =       0.0000

-> F = 17.51 >= 10: instruments are not weak (rule of thumb).
    Minimum eigenvalue statistic 16.72 is above the Stock-Yogo
    critical value 16.38 (2SLS Wald test of nominal size 5% with
    true size at most 10%); these values assume homoskedastic
    errors.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


第一阶段 F 是 17.5，超过了常用的经验门槛 10。不过 10 这个门槛近年受到不少质疑。`sp.iv.iv_diag` 把现在审稿人常要求的几项诊断打包在一起。

| 指标 | 说明 |
| --- | --- |
| Olea-Pflueger 有效 F | 对异方差和聚类稳健的第一阶段 F |
| tF 修正 | Lee, McCrary, Moreira and Porter (2022)。根据第一阶段 F 调大 t 检验的临界值 |
| Anderson-Rubin 置信集 | 不依赖工具变量强度，任何 F 下都有正确的覆盖率 |

In [11]:
diag = sp.iv.iv_diag(
    card, y="lwage", endog="educ", instruments=["nearc4"],
    exog=["exper", "expersq", "black", "south", "smsa"],
    n_boot=500, random_state=0,
)
print(diag.summary())

Modern IV reporting bundle  (sp.iv.iv_diag)
  endog                       : educ
  instruments                 : nearc4
  exogenous controls          : exper, expersq, black, south, smsa
  N = 3010, k_endog = 1, k_instr = 1, k_exog = 5
  level                       : 95 %
----------------------------------------------------------------------
Strength:
  First-stage F (classical)   :    16.7176
  Olea–Pflueger effective F   :    17.5133
  Kleibergen–Paap rk LM       :    17.3414   p = 0.0000
  Kleibergen–Paap rk Wald F   :    17.5133
  LMMP 2022 tF adjusted crit. :     2.7160   (F = 16.72)
----------------------------------------------------------------------
Point estimate:
  2SLS                        :     0.1323   SE = 0.0486   t = 2.723   p = 0.0065
  Wald  95% CI                : [0.0371, 0.2275]
  tF-corrected CI             : [0.0003, 0.2642]
  Pairs bootstrap (B=500)  : SE = 0.0526,  CI [0.0421, 0.2575]
----------------------------------------------------------------------
Wea

三种区间可以放在一起比较。

- 常规 Wald 区间是 [0.037, 0.228]。
- Anderson-Rubin 置信集是 [0.038, 0.261]，右端更宽一些，仍然不含 0。
- tF 修正后的区间是 [0.000, 0.264]，下界几乎贴着 0。

结论是，教育回报率为正这一点在弱工具变量稳健的推断下基本站得住，但证据的强度比常规 t 检验显示的 p = 0.007 要弱。估计值的不确定性很大，区间从 4% 一直延伸到 26%。

## 8. 过度识别

数据里还有一个候选工具变量 `nearc2`，表示附近有没有两年制大学。工具变量个数多于内生变量个数时，模型是**过度识别**的。这带来一个好处，可以检验工具变量之间是否互相矛盾。

In [12]:
iv2 = sp.ivreg(f"lwage ~ (educ ~ nearc4 + nearc2) + {controls}", data=card, robust="hc1")
_ = sp.estat(iv2, "overid")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Sargan/Hansen J over-identification test
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  H0: All instruments are valid (exclusion restrictions hold)
  H1: At least one instrument is invalid

  chi2(1)    =       2.6532
  p-value    =       0.1033

-> Cannot reject H0 at 5%: over-identifying restrictions are
    satisfied. Instruments appear valid.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Hansen J 检验的原假设是所有工具变量都有效。p = 0.10，在 5% 水平上不拒绝。

解读时要小心两点。

- 这个检验的逻辑是「如果其中一个工具变量有效，另一个给出的估计是否与它一致」。两个工具变量如果以同样的方式失效，检验发现不了。
- 不拒绝原假设不等于证明了工具变量有效，功效可能很低。

加入第二个工具变量还有代价。看下面的对比表。

In [13]:
# LIML 在工具变量较弱或较多时偏差比 2SLS 小
liml = sp.iv(f"lwage ~ (educ ~ nearc4 + nearc2) + {controls}", data=card, method="liml", robust="hc1")

print(sp.regtable(
    ols, iv, iv2, liml,
    keep=["educ"],
    model_labels=["OLS", "2SLS", "2SLS (2 IVs)", "LIML (2 IVs)"],
))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                             OLS          2SLS  2SLS (2 IVs)  LIML (2 IVs)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
educ                   0.0740***     0.1323***     0.1608***     0.1746***
                        (0.0036)      (0.0486)      (0.0486)      (0.0579)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
First-stage F                            17.51          9.72          9.64
KP rk Wald F                                                          9.72
Hansen J p-value                                       0.103         0.113
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
N                          3,010         3,010         3,010         3,010
R²                         0.291         0.225         0.146         0.096
Adj. R²                    0.289                                          
F               

`sp.regtable` 相当于 Stata 的 `esttab`，IV 模型的第一阶段 F 和 Hansen J 的 p 值会自动列在表下方。

加入 `nearc2` 之后，第一阶段 F 从 17.5 **降到** 9.7。`nearc2` 本身对教育年限的解释力很弱，多加一个弱工具变量稀释了整体强度。此时 2SLS 和 LIML 的点估计也拉开了距离（0.161 对 0.175）。在工具变量很强时两者应该几乎相同，差距变大本身就是弱工具变量的一个信号。

这个例子说明，工具变量不是越多越好。一个强的工具变量通常胜过一个强的加几个弱的。

## 9. 估计的是谁的效应

到目前为止我们默认教育回报率 $\beta$ 对所有人相同。如果每个人的回报率不一样，IV 估计的是什么？

Angrist, Imbens and Rubin (1996) 的回答是，在工具变量有效并且满足**单调性**（工具变量不会使任何人的处理水平朝相反方向变化）的条件下，IV 估计的是**依从者**的平均效应，称为局部平均处理效应（LATE）。

在这个例子里，依从者是那些「附近有大学就会多读书，没有就不读」的人。他们很可能来自经济条件较差的家庭，上学成本对他们的决定影响最大。那些无论如何都会上大学的人，以及无论如何都不会上的人，对 IV 估计没有贡献。

这为「IV 估计值比 OLS 大」提供了几种可能的解释。

1. **测量误差**。OLS 因为教育年限的测量误差而被低估，IV 修正了它。
2. **效应异质性**。依从者的教育回报率高于总体平均。受到信贷约束的人边际回报可能更高。
3. **工具变量并非完全有效**。`nearc4` 可能通过教育以外的渠道与工资相关。第一阶段很弱时，哪怕很小的直接效应也会被放大。

这三种解释数据本身分不开。报告 IV 结果时，应该说明估计量对应的是哪个子总体，而不是把它当作「所有人的教育回报率」。

另外，稳健的内生性检验（`sp.estat(iv, "endogenous")`，也已附在第 6 节的结果下面）的 p 值是 0.21，不能拒绝 `educ` 外生的原假设。这不代表 OLS 是对的。IV 的标准误太大，检验没有足够的功效区分 0.074 和 0.132。

## 10. 排除限制不完全成立怎么办

排除限制要求 `nearc4` 对工资的直接效应恰好为 0。Conley, Hansen and Rossi (2012) 建议放松这个要求。假设直接效应 $\gamma$ 不一定为 0，但落在一个研究者认为合理的范围里，看结论如何变化。

$$
\log(\text{wage}) = \beta\,\text{educ} + \gamma\,\text{nearc4} + X'\delta + u, \qquad \gamma \in [\gamma_{\min},\, \gamma_{\max}].
$$

简约式里 `nearc4` 的总效应是 0.045。下面假设其中最多有 0.02 是不经过教育的直接效应，方向可正可负。

In [14]:
pe = sp.iv.plausibly_exogenous_uci(
    y="lwage", endog="educ", instruments=["nearc4"],
    gamma_grid=np.linspace(-0.02, 0.02, 9),
    exog=["exper", "expersq", "black", "south", "smsa"],
    data=card,
)
print(pe.summary())

# 每个假想的直接效应 gamma 对应的 IV 估计值
pd.DataFrame({
    "gamma": np.ravel(pe.gamma_grid),
    "beta_iv": pe.beta_at_gamma,
    "se": pe.se_at_gamma,
}).round(4)

Plausibly exogenous IV — UCI (union of CIs)
------------------------------------------------
  Point estimate (γ=0) :     0.1323   SE=0.0492
  Sensitivity CI (95%): [-0.0193, 0.2998]
  γ grid size          : 9


,gamma,beta_iv,se
0,-0.020,0.1916,0.0552
1,-0.015,0.1768,0.0534
2,-0.010,0.1619,0.0518
3,-0.005,0.1471,0.0504
4,0.000,0.1323,0.0492
5,0.005,0.1175,0.0483
6,0.010,0.1026,0.0476
7,0.015,0.0878,0.0472
8,0.020,0.0730,0.0471


允许直接效应在 ±0.02 之间之后，置信区间变成 [−0.019, 0.300]，明显变宽，并且包含了 0。

原因可以直接算出来。直接效应 $\gamma$ 会使 IV 估计量偏离 $\gamma / \pi_1$。第一阶段系数 $\pi_1$ 只有 0.34，所以 0.02 的直接效应就能使估计值移动约 0.06，差不多是点估计的一半。**第一阶段越弱，结论对排除限制的违背越敏感**。这是弱工具变量的另一重代价，它和第 7 节的统计推断问题是两回事。

## 11. 小结

报告 IV 结果时可以对照这份清单。

1. **讲清楚识别故事**。工具变量为什么相关，为什么外生，为什么只通过内生变量起作用。这一步数据帮不了你。
2. **报告第一阶段和简约式**。简约式不显著的时候，IV 估计通常不可信。
3. **报告工具变量强度**。第一阶段 F 或有效 F（`sp.estat(iv, "firststage")`，`sp.iv.iv_diag`）。
4. **报告弱工具变量稳健的区间**。Anderson-Rubin 置信集或 tF 修正，尤其是 F 不够大时。
5. **不要堆砌工具变量**。过度识别时报告 Hansen J（`sp.estat(iv, "overid")`），并用 LIML 复核（`sp.iv(..., method="liml")`）。
6. **说明估计的是谁的效应**。LATE 对应依从者，不是总体。
7. **对排除限制做敏感性分析**（`sp.iv.plausibly_exogenous_uci`）。
8. **永远和 OLS 并排报告**（`sp.regtable`），并解释两者差异的来源。

延伸阅读见 `docs/guides/choosing_iv_estimator.md`。

### 本教程用到的方法的文献

下面的 BibTeX 条目直接取自 StatsPAI 核验过的 `paper.bib`。

In [15]:
keys = [
    "card1995using",
    "angrist1996identification",
    "staiger1997instrumental",
    "stock2005testing",
    "olea2013robust",
    "lee2022valid",
    "anderson1949estimation",
    "conley2012plausibly",
]
print(sp.bibtex(keys))

@incollection{card1995using,
  title={Using Geographic Variation in College Proximity to Estimate the Return to Schooling},
  author={Card, David},
  booktitle={Aspects of Labour Market Behaviour: Essays in Honour of {John Vanderkamp}},
  editor={Christofides, Louis N. and Grant, E. Kenneth and Swidinsky, Robert},
  publisher={University of Toronto Press},
  address={Toronto},
  pages={201--222},
  year={1995},
  isbn={0802028640},
  annote={Published version of NBER Working Paper 4483 (doi 10.3386/w4483), which is deliberately not kept as a separate entry (one record per work, tests/test_citation_integrity.py). Volume record (title, editors, publisher, year, ISBN) verified 2026-09-04 via OpenLibrary and the NBER w4483 "Published Versions" note; chapter page range from the previously verified JSS manuscript bibliography. No DOI exists for this chapter.}
}

@article{angrist1996identification,
  title={Identification of Causal Effects Using Instrumental Variables},
  author={Angrist, Jos